# Stage 6 — Clustering Job Postings into Role Archetypes

**Research question:** do distinct job role archetypes exist beyond traditional job titles?

Each job is represented by its skills: a multi-hot vector over the 100-skill vocabulary, as in Stage 5. Two methods are applied:
- **K-Means** for k = 3..10, with k chosen by silhouette score (elbow curve shown too)
- **DBSCAN**, with eps tuned from the k-distance graph

Both are scored with **silhouette** (target > 0.50) and **Davies-Bouldin** (target < 1.0). Each K-Means cluster is then profiled by its defining skills, common titles, and salary, and named from its skill profile.

The logic lives in `src/clustering.py`. To run without the notebook: `python -m src.clustering`.

## Step 1 — Setup

In [ ]:
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image


def find_root(start: Path) -> Path:
    """Return the first directory at or above `start` that contains CLAUDE.md."""
    for candidate in [start, *start.parents]:
        if (candidate / "CLAUDE.md").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root (no CLAUDE.md found above cwd).")


PROJECT_ROOT = find_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT))

from src import clustering as clu

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Step 2 — Build skill vectors

Jobs with **no** matched skill (about 1.5%) would all be the same all-zero vector and form an artificial cluster, so they're set aside and labelled `-1` ("No matched skills").

In [ ]:
jobs, matrix = clu.load_skill_matrix(PROJECT_ROOT)
has_skills = (matrix.sum(axis=1) > 0).to_numpy()
X = matrix.to_numpy(dtype=float)[has_skills]
n_distinct = len(np.unique(X, axis=0))
print(f"{len(jobs):,} jobs; clustering {len(X):,} with >= 1 skill; {n_distinct:,} distinct skill sets")

## Step 3 — K-Means, k = 3 to 10

Silhouette is computed on a fixed random sample of 10,000 jobs, because the exact score is O(n²).

In [ ]:
sweep, km_models = clu.kmeans_sweep(X)
best_k = int(sweep.loc[sweep["silhouette"].idxmax(), "k"])
km_fit = km_models[best_k].labels_
clu.plot_elbow(sweep, best_k, OUTPUT_DIR / "06_elbow_plot.png")
display(Image(filename=str(OUTPUT_DIR / "06_elbow_plot.png"), width=950))
sweep.round(4)

The inertia curve falls almost linearly, with **no elbow**. Silhouette is highest at k = 3, but every value is below 0.04. In other words, jobs sit nearly as close to other clusters as to their own. Nearly 27,000 distinct skill sets among 33,675 jobs form a continuum rather than separate groups.

## Step 4 — DBSCAN with eps from the k-distance graph

With `min_samples = 20`, each job's distance to its 20th nearest neighbour is sorted and plotted. On binary vectors, Euclidean distance is √(number of differing skills), so the graph is a **staircase** of steps at √1, √2, √3, …

Each step between the 10th and 90th percentile is a candidate `eps`. The chosen value has the highest silhouette among runs with at least 2 clusters and at most 50% noise.

In [ ]:
kdist = clu.k_distances(X)
dbscan, dbscan_table = clu.tune_dbscan(X, kdist)
clu.plot_kdistance(kdist, dbscan["eps"], OUTPUT_DIR / "06_kdistance_plot.png")
display(Image(filename=str(OUTPUT_DIR / "06_kdistance_plot.png"), width=700))
dbscan_table.round(4)

Every setting gives **one dominant cluster**, holding 95-100% of clustered jobs, plus noise and a few tiny clusters. With a small eps, most jobs become noise. With a larger eps, everything merges. DBSCAN finds no dense, separated regions, just one connected mass of overlapping skill sets.

Its silhouette and Davies-Bouldin scores come from those tiny side clusters. They say little about archetypes.

## Step 5 — Evaluation against targets

In [ ]:
km_scores = sweep.loc[sweep["k"] == best_k, ["silhouette", "davies_bouldin"]].iloc[0]
evaluation = pd.DataFrame([
    {"method": f"K-Means (k={best_k})", "clusters": best_k, "noise_share": 0.0, **km_scores.to_dict()},
    {"method": f"DBSCAN (eps={dbscan['eps']:.2f})", "clusters": dbscan["n_clusters"],
     "noise_share": dbscan["noise_share"], "silhouette": dbscan["silhouette"],
     "davies_bouldin": dbscan["davies_bouldin"]},
])
evaluation["silhouette > 0.50"] = evaluation["silhouette"] > clu.SILHOUETTE_TARGET
evaluation["Davies-Bouldin < 1.0"] = evaluation["davies_bouldin"] < clu.DAVIES_BOULDIN_TARGET
evaluation.round(4)

**Sensitivity check:** would a denser representation separate jobs better? TF-IDF weighting reduced to 10 dimensions (SVD) and length-normalized raises silhouette to about 0.14 at the same k. That's better, but still far below 0.50, so the conclusion doesn't depend on the encoding.

In [ ]:
comparison = clu.compare_representations(X, best_k)
comparison.round(4)

## Step 6 — Cluster profiles

For each K-Means cluster:
- **Defining skills:** ranked by lift (share in the cluster ÷ share overall). A skill must appear in ≥ 15% of the cluster's jobs and be over-represented there (lift > 1).
- **Most common job titles**
- **Salary:** mean, median, and Low/Mid/High mix
- **Name:** taken from the skill theme (`clu.SKILL_THEMES`) that best matches the defining skills. A cluster with no over-represented skill is the **generalist** group, whose postings list fewer skills overall.

In [ ]:
km_labels = np.full(len(jobs), clu.NO_SKILLS_LABEL)
km_labels[has_skills] = km_fit
db_labels = np.full(len(jobs), clu.NO_SKILLS_LABEL)
db_labels[has_skills] = dbscan["labels"]

tables = clu.defining_skills(matrix, km_labels)
names = clu.name_clusters(tables)
profiles = clu.build_profiles(jobs, matrix, km_labels, names, tables)
profiles_text = clu.format_profiles(profiles, f"SkillMap — K-Means cluster profiles (k = {best_k})")
(OUTPUT_DIR / "06_cluster_profiles.txt").write_text(profiles_text, encoding="utf-8")
print(profiles_text)

## Step 7 — Save cluster assignments

`06_cluster_labels.csv` has one row per job with its K-Means cluster and name, its DBSCAN cluster (`-1` = noise or no matched skills), and its salary.

In [ ]:
labels_out = pd.DataFrame({
    "job_id": jobs["job_id"],
    "job_title": jobs["job_title"],
    "kmeans_cluster": km_labels,
    "kmeans_cluster_name": [names.get(c, clu.NO_SKILLS_NAME) for c in km_labels],
    "dbscan_cluster": db_labels,
    "salary_tier": jobs["salary_tier"],
    "normalized_salary": jobs["normalized_salary"],
})
labels_out.to_csv(OUTPUT_DIR / "06_cluster_labels.csv", index=False)
labels_out["kmeans_cluster_name"].value_counts()

## Step 8 — PCA visualization

The skill vectors are projected onto their first two principal components. 8,000 jobs are sampled, with slight jitter because many jobs share identical binary vectors. The two components explain only about 12% of the variance, so the plot shows a single continuous cloud. The K-Means clusters look like slices of that cloud, and DBSCAN colours almost all of it as one cluster.

In [ ]:
explained = clu.plot_pca(X, km_fit, names, dbscan["labels"], OUTPUT_DIR / "06_clusters_pca.png")
Image(filename=str(OUTPUT_DIR / "06_clusters_pca.png"), width=1000)

## Step 9 — Save the summary

In [ ]:
summary = clu.build_summary(len(jobs), len(X), sweep, best_k, km_scores.to_dict(), dbscan, dbscan_table,
                            comparison, profiles, explained)
(OUTPUT_DIR / "06_summary.txt").write_text(summary, encoding="utf-8")
print(summary)

## Findings

- **Targets not met.** K-Means scores silhouette 0.03 and Davies-Bouldin 4.2. DBSCAN's best qualifying run scores 0.17 and 1.04, with one cluster holding 99.9% of clustered jobs.
- **This is a property of the data, not of the tuning.** Every k, every eps candidate, and a denser TF-IDF/SVD representation (silhouette about 0.14) point the same way. With a vocabulary of mostly general skills, job postings form a skill continuum, not well-separated groups.
- **The K-Means partition still has meaningful salary differences:**
  - **Management & Leadership** (28% of jobs): project management, leadership, planning, collaboration, reporting. Mean **$114k**, 45% High tier.
  - **Education & Training** (27%): training, high school diploma, safety, customer service. Mean **$80k**, 47% Low tier.
  - **Generalist** (44%): postings listing fewer skills (5.4 on average). Mean $95k, spread evenly across tiers.
- **Answer to the research question:** the skill data shows broad *tendencies* (a leadership-heavy, higher-paid profile versus a training/credential-heavy, lower-paid one), not crisp archetypes beyond job titles. More specific archetypes (e.g. data engineering) would need a larger, more technical skill vocabulary. Only python and sql are in the current top 100.